## Traffic Assignments for system ownership outage scenarios

In this notebook ownership system outages are simulated by removing them from the network and the traffic assignment is run iteratively for each system ownership being removed and the results are saved


In [1]:
import geopandas as gpd
from shapely.geometry import Point, MultiPoint, LineString, MultiLineString
from shapely.ops import split
from geopy.distance import geodesic
import pandas as pd
import os
import re
import networkx as nx
from shapely.geometry import Point, LineString, MultiPolygon, Polygon
from shapely.ops import unary_union # For distance calc
from pyproj import Geod, CRS, Transformer # For distance calc projection
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.sample import sample_gen
import numpy as np
import math
import logging
from contextlib import contextmanager
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
import time
from geopy.geocoders import MapBox
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import pickle
import requests # Not used currently
import json # Not used currently
#Import required packages and modules
#import qgis
import PyQt5
from aequilibrae import Graph, AequilibraeMatrix, TrafficClass, TrafficAssignment
from aequilibrae.matrix import AequilibraeMatrix
from aequilibrae.paths import Graph
from aequilibrae.paths import TrafficAssignment
from aequilibrae.paths.traffic_class import TrafficClass
#from qgis.core import QgsVectorLayer, QgsField, QgsFeature, QgsPointXY, QgsProject, QgsGeometry, QgsVectorFileWriter
from PyQt5.QtCore import QVariant 
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ast

In the code, the ```split_antimeridian_line```, ```plot_traffic```, and ```process_scenario``` functions are reused from the code which runs the baseline scenario found in the notebook titled "TA Baseline & Plotting" within the repository.

In [2]:
# 1. Load data
od_data = pd.read_csv("OD_filtered.csv")

def split_antimeridian_line(start, end):
    lon1, lat1 = start
    lon2, lat2 = end
    
    # Check if line crosses antimeridian
    if abs(lon1 - lon2) <= 180:
        # No crossing, return as is
        return [((lon1, lat1), (lon2, lat2))]
    
    # Crossing detected: split into two segments
    crossing_lon = 180 if lon1 > 0 else -180
    
    # Linear interpolation to find crossing latitude
    if lon2 != lon1:
        ratio = (crossing_lon - lon1) / (lon2 - lon1)
    else:
        ratio = 0.5  # Arbitrary if same longitude
    
    crossing_lat = lat1 + ratio * (lat2 - lat1)
    
    # First segment: start to crossing point
    segment1_end = (crossing_lon, crossing_lat)
    # Second segment: crossing point (wrapped) to end
    if lon1 > lon2:
        segment2_start = (crossing_lon - 360, crossing_lat)
    else:
        segment2_start = (crossing_lon + 360, crossing_lat)
    
    return [((lon1, lat1), segment1_end), (segment2_start, (lon2, lat2))]

def plot_traffic(merged_data, nodes, traffic_col='traffic_all', title=None):
    fig, ax = plt.subplots(figsize=(15, 10))
    max_flow = merged_data[traffic_col].max()
    norm = mcolors.Normalize(vmin=0, vmax=max_flow)
    cmap = plt.cm.viridis

    for _, link in merged_data.iterrows():
        start_node = nodes[nodes['Hop_ID'] == link['start_point_idx']]
        end_node = nodes[nodes['Hop_ID'] == link['end_point_idx']]
        
        if not start_node.empty and not end_node.empty:
            # Capital link detection and conditional plotting
            is_capital_link = (link['Start Node'].startswith('capital_') 
                              and link['End Node'].startswith('capital_'))
            
            if is_capital_link:
                if link[traffic_col] <= 0:
                    continue  # Skip non-active capital links
                line_color = 'red'
            else:
                line_color = cmap(norm(link[traffic_col]))

            width = 2.5 + 14 * (link[traffic_col] / max_flow if max_flow > 0 else 0)

            start_coords = (start_node['Longitude'].values[0], start_node['Latitude'].values[0])
            end_coords = (end_node['Longitude'].values[0], end_node['Latitude'].values[0])

            # Split line if crossing antimeridian
            segments = split_antimeridian_line(start_coords, end_coords)
            for seg_start, seg_end in segments:
                ax.plot(
                    [seg_start[0], seg_end[0]],
                    [seg_start[1], seg_end[1]],
                    color=line_color,
                    linewidth=width,
                    alpha=0.7,
                    zorder=1
                )

    # Plot nodes (same as your original code)
    landing_points = nodes[nodes['Node ID'].str.startswith('pt')]
    ax.scatter(
        landing_points['Longitude'], landing_points['Latitude'],
        color='blue', s=60, marker='o', edgecolor='black', linewidth=1,
        label='Landing Points', zorder=3
    )
    grid_points = nodes[nodes['Node ID'].str.startswith('grid')]
    ax.scatter(
        grid_points['Longitude'], grid_points['Latitude'],
        color='green', s=40, marker='s', edgecolor='black', linewidth=1,
        label='Junction Points', zorder=2
    )
    capitals = nodes[nodes['Node ID'].str.startswith('capital')]
    ax.scatter(
        capitals['Longitude'], capitals['Latitude'],
        color='red', s=100, marker='^', edgecolor='black', linewidth=1,
        label='Capital Cities', zorder=4
    )
    for _, capital in capitals.iterrows():
        city_name = capital['Name'].split(',')[0]
        ax.annotate(
            city_name,
            (capital['Longitude'], capital['Latitude']),
            xytext=(5, 5), textcoords='offset points',
            fontsize=9, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8)
        )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label('Traffic Flow Volume', fontsize=12)
    ax.set_title(title or f'Traffic Flows: {traffic_col}', fontsize=16)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='lower right')

    # Add text showing top flows
    top_flows = merged_data.sort_values(traffic_col, ascending=False).head(3)
    flow_text = "Top Flows:\n"
    for i, (_, flow) in enumerate(top_flows.iterrows(), 1):
        if 'Start Name' in flow and 'End Name' in flow:
            start = flow['Start Name'].split(',')[0] if ',' in flow['Start Name'] else flow['Start Name']
            end = flow['End Name'].split(',')[0] if ',' in flow['End Name'] else flow['End Name']
            flow_text += f"{i}. {start} → {end}: {flow[traffic_col]:.1f}\n"
    ax.text(0.02, 0.02, flow_text, transform=ax.transAxes, 
            bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5'),
            fontsize=9)

    plt.tight_layout()
    plt.show()

def print_active_links(merged_data, traffic_col):
    active_links = merged_data[merged_data[traffic_col] > 0]['link_id']
    print(f"Active links for {traffic_col} (traffic > 0):")
    for link_id in active_links:
        print(link_id)
    return active_links.tolist()

In [3]:
def process_scenario(nodes_df, links_df, od_data):
    # 1. Set relevant_countries to the unique, non-null values in the 'Country' column
    relevant_countries = set(nodes_df['Country'].dropna().unique())

    # 2. Read capitals.csv and filter for relevant countries
    capitals_raw = pd.read_csv("capitals.csv")
    capitals_filtered = capitals_raw[capitals_raw['Country'].isin(relevant_countries)]

    # 3. Create dictionary mapping countries to their capitals and coordinates
    country_capitals = {
        row['Country']: (row['Capital'], row['Longitude'], row['Latitude'])
        for _, row in capitals_filtered.iterrows()
    }

    # 4. Create capital nodes
    max_hop = nodes_df['Hop_ID'].max()
    capital_nodes = []
    for idx, (country, (capital, lon, lat)) in enumerate(country_capitals.items(), 1):
        capital_nodes.append({
            'Node ID': f"capital_{country.replace(' ', '_')}",
            'Name': f"{capital}, {country}",
            'Longitude': lon,
            'Latitude': lat,
            'Hop_ID': max_hop + idx
        })
    capitals_df = pd.DataFrame(capital_nodes)
    nodes_df = pd.concat([nodes_df, capitals_df], ignore_index=True)

    # 5. Connect landing points to capitals
    capital_links = []
    for _, lp in nodes_df[nodes_df['Node ID'].str.startswith('pt')].iterrows():
        country = lp['Country']  # Use the country column directly
        if country in country_capitals:
            capital = capitals_df[capitals_df['Name'].str.endswith(country)].iloc[0]
            capital_links.append({
                'Edge ID': f"{lp['Node ID']}-{capital['Node ID']}",
                'Start Node': lp['Node ID'],
                'Start Name': lp['Name'],
                'Start Coordinates': f"({lp['Longitude']}, {lp['Latitude']})",
                'End Node': capital['Node ID'],
                'End Name': capital['Name'],
                'End Coordinates': f"({capital['Longitude']}, {capital['Latitude']})",
                'distance': 50,
                'start_point_idx': lp['Hop_ID'],
                'end_point_idx': capital['Hop_ID']
            })
    if capital_links:
        capital_links_df = pd.DataFrame(capital_links)
        links_df = pd.concat([links_df, capital_links_df], ignore_index=True)

    # Fill empty distance_km with distance
    links_df.loc[links_df['distance_km'].isna(), 'distance_km'] = links_df['distance']

    # Drop the distance column
    links_df = links_df.drop(columns=['distance'], errors='ignore')

    # Create capital-to-capital links
    capital_to_capital_links = []
    for i, row_i in capitals_df.iterrows():
        for j, row_j in capitals_df.iterrows():
            if i != j:  # Avoid self-connections
                capital_to_capital_links.append({
                    'Edge ID': f"{row_i['Node ID']}-{row_j['Node ID']}",
                    'Start Node': row_i['Node ID'],
                    'Start Name': row_i['Name'],
                    'Start Coordinates': f"({row_i['Longitude']}, {row_i['Latitude']})",
                    'End Node': row_j['Node ID'],
                    'End Name': row_j['Name'],
                    'End Coordinates': f"({row_j['Longitude']}, {row_j['Latitude']})",
                    'distance_km': 1e6,  # Fixed distance between capitals
                    'start_point_idx': row_i['Hop_ID'],
                    'end_point_idx': row_j['Hop_ID']
                })
    if capital_to_capital_links:
        capital_to_capital_links_df = pd.DataFrame(capital_to_capital_links)
        links_df = pd.concat([links_df, capital_to_capital_links_df], ignore_index=True)

    # 6. Create network DataFrame
    network = pd.DataFrame({
        'link_id': range(1, len(links_df)+1),
        'a_node': links_df['start_point_idx'].astype(int),
        'b_node': links_df['end_point_idx'].astype(int),
        'direction': 0,
        'capacity': 13570,
        'free_flow_time': links_df['distance_km'] / (299792 * 0.75),  # using 75% the speed of light in km/h
    })

    # NOTE: The following code assumes you have defined Graph, TrafficAssignment, AequilibraeMatrix, TrafficClass classes
    # If you do not, you will need to import them or define them.
    g = Graph()
    g.network = network
    g.status = 'OK'
    centroids = capitals_df['Hop_ID'].unique().astype(int)
    g.prepare_graph(centroids)
    g.set_graph("free_flow_time")

    # 8. Create mapping from country name to capital Hop_ID
    country_to_hop = {country: capitals_df[capitals_df['Name'].str.contains(country)]['Hop_ID'].values[0] 
                      for country in country_capitals.keys()}

    # 9. Automatically detect all countries present in OD data and in the mapping
    all_countries = set(od_data['Country1'].str.strip()) | set(od_data['Country2'].str.strip())
    countries_of_interest = sorted([c for c in all_countries if c in country_to_hop])

    # 10. Prepare OD matrices for each country of interest and for 'others'
    od_matrices = {country: np.zeros((len(centroids), len(centroids)), dtype=np.float64) for country in countries_of_interest}
    od_matrices['others'] = np.zeros((len(centroids), len(centroids)), dtype=np.float64)

    for _, row in od_data.iterrows():
        orig_country = row['Country1'].strip()
        dest_country = row['Country2'].strip()
        flow = row['Flow']
        try:
            orig_hop = country_to_hop[orig_country]
            dest_hop = country_to_hop[dest_country]
            orig_idx = np.where(centroids == orig_hop)[0][0]
            dest_idx = np.where(centroids == dest_hop)[0][0]
            if orig_country in countries_of_interest:
                od_matrices[orig_country][orig_idx, dest_idx] = flow
            else:
                od_matrices['others'][orig_idx, dest_idx] = flow
        except KeyError:
            pass

    # 11. Create Aequilibrae matrices and add as classes
    matrices = {}
    assig = TrafficAssignment()
    for country, matrix_data in od_matrices.items():
        mat = AequilibraeMatrix()
        mat.create_empty(zones=len(centroids), matrix_names=['matrix'], memory_only=True)
        mat.index[:] = centroids
        mat.matrix['matrix'][:, :] = matrix_data
        mat.computational_view(['matrix'])
        matrices[country] = mat
        assig.add_class(TrafficClass(country, g, mat))

    # 12. Assignment settings
    assig.set_vdf('BPR')
    assig.set_vdf_parameters({'alpha': 0.15, 'beta': 1.0})
    assig.set_capacity_field('capacity')
    assig.set_time_field('free_flow_time')
    assig.set_algorithm('bfw')
    assig.max_iter = 180
    assig.rgap_target = 1e-4

    assig.execute()
    results = assig.results()

    # 13. Rename columns for clarity and add total traffic column
    for i, country in enumerate(matrices.keys()):
        col_idx = 2 + 3*i  # traffic column index pattern
        if col_idx < len(results.columns):
            results.columns.values[col_idx] = f'traffic_{country.replace(" ", "_")}'
    traffic_cols = [col for col in results.columns if col.startswith('traffic_')]
    results['traffic_all'] = results[traffic_cols].sum(axis=1)

    # Rename by position
    traffic = results
    traffic = traffic.reset_index()
    if 'link_id1' not in traffic.columns:
        traffic = traffic.reset_index().rename(columns={'index': 'link_id1'})

    # Link the traffic data with the network links
    merged_data = pd.merge(
        links_df,
        traffic,
        left_index=True,
        right_on='link_id1',
        how='left'
    )

    # 1. Get list of origin countries from OD data (excluding 'others')
    countries = [c for c in od_matrices.keys() if c != 'others']

    # 2. Map capital node IDs to country names
    capital_to_country = {
        row['Node ID']: row['Name'].split(', ')[-1]
        for _, row in capitals_df.iterrows()
    }

    # 3. Identify all capital-to-capital links in the network
    capital_links_mask = (links_df['Start Node'].str.startswith('capital_')) & \
                         (links_df['End Node'].str.startswith('capital_'))
    capital_link_ids = links_df[capital_links_mask].index

    # 4. Create country links dataframe
    country_links_data = []
    for country in countries:
        traffic_col = f'traffic_{country.replace(" ", "_")}'
        active_links = results[results[traffic_col] > 0].index

        # Existing metrics
        system_ids = links_df.loc[active_links, 'system_ID'].unique()
        avg_voc = results.loc[active_links, 'VOC_max'].mean() if 'VOC_max' in results.columns else None
        avg_delay = results.loc[active_links, 'Delay_factor_Max'].mean() if 'Delay_factor_Max' in results.columns else None
        avg_traffic = results.loc[active_links, 'traffic_all'].mean() if 'traffic_all' in results.columns else None

        # Capital link detection
        capital_mask = (links_df.loc[active_links, 'Start Node'].str.startswith('capital_')) & \
                       (links_df.loc[active_links, 'End Node'].str.startswith('capital_'))
        uses_capital = capital_mask.any()
        num_capital = capital_mask.sum()

        # Get unique destination countries via capital-to-capital links
        capital_links_used = links_df.loc[active_links.intersection(capital_link_ids)]
        dest_countries = set()
        for _, link in capital_links_used.iterrows():
            start_country = capital_to_country.get(link['Start Node'])
            end_country = capital_to_country.get(link['End Node'])
            if start_country and end_country:
                # For origin country, the other country is the destination (but direction is not always clear)
                dest_countries.add(end_country)
                dest_countries.add(start_country)  # both, since direction is not always clear
        capital_link_countries = ', '.join(sorted(dest_countries)) if dest_countries else None

        # Calculate total traffic assigned via capital-to-capital links for this country
        capital_traffic = results.loc[active_links.intersection(capital_link_ids), traffic_col].sum()

        country_links_data.append({
            'Origin Country': country,
            'Links Used': active_links.tolist(),
            'System_IDs': system_ids.tolist(),
            'Num Unique System_IDs': len(system_ids),
            'Num Links Used': len(active_links),
            'Avg VOC_max': avg_voc,
            'Avg Delay_factor_Max': avg_delay,
            'Avg traffic_all': avg_traffic,
            'Uses Capital Link': uses_capital,
            'Num Capital Links Used': num_capital,
            'Capital Link Countries': capital_link_countries,  # Countries whose capitals are connected by capital-to-capital links
            'Capital-to-Capital Traffic': capital_traffic     # Total traffic assigned via capital-to-capital links for this country
        })

    # 5. Create dataframe
    country_links = pd.DataFrame(country_links_data)

    # 6. Calculate averages for all numeric columns
    avg_values = {
        'Num Unique System_IDs': country_links['Num Unique System_IDs'].mean(),
        'Num Links Used': country_links['Num Links Used'].mean(),
        'Avg VOC_max': country_links['Avg VOC_max'].mean(),
        'Avg Delay_factor_Max': country_links['Avg Delay_factor_Max'].mean(),
        'Avg traffic_all': country_links['Avg traffic_all'].mean(),
        'Num Capital Links Used': country_links['Num Capital Links Used'].mean(),
        'Capital-to-Capital Traffic': country_links['Capital-to-Capital Traffic'].mean()
    }

    # 7. Add averages row
    avg_row = pd.DataFrame({
        'Origin Country': ['Average'],
        'Links Used': [None],
        'System_IDs': [None],
        **{k: [v] for k, v in avg_values.items()},
        'Uses Capital Link': [None],
        'Capital Link Countries': [None]
    })

    # 8. Combine and format
    final_df = pd.concat([country_links, avg_row], ignore_index=True)
    final_df = final_df[[
        'Origin Country', 
        'Num Links Used',
        'Num Unique System_IDs',
        'Avg VOC_max',
        'Avg Delay_factor_Max',
        'Avg traffic_all',
        'Uses Capital Link',
        'Num Capital Links Used',
        'Capital Link Countries',
        'Capital-to-Capital Traffic',
        'System_IDs',
        'Links Used'
    ]]

    # Formatting for readability
    final_df['System_IDs'] = final_df['System_IDs'].apply(
        lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x
    )
    final_df['Links Used'] = final_df['Links Used'].apply(
        lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x
    )

    return final_df, merged_data, nodes_df, links_df

Here, the ownership information file is read ```OwnershipInformationCables.xlsx``` and the affiliation is tagged to each cable system

In [4]:
ownership_df = pd.read_excel('OwnershipInformationCables.xlsx')
nodes = pd.read_csv("2024FullNetworkNodes.csv")
links = pd.read_csv("2024FullNetworkEdges.csv")
ownership_df = ownership_df[ownership_df['year_WFS'] >= 2025]
ownership_df = ownership_df.drop(['ID', 'owner', 'length_km', 'segment_ID'], axis=1)

# Get all column names except 'year_RFS' and 'year_WFS'
cols_to_consider = [col for col in ownership_df.columns if col not in ['year_RFS', 'year_WFS']]
# Drop duplicates based on those columns
ownership_df = ownership_df.drop_duplicates(subset=cols_to_consider)
unique_affiliations = ownership_df['geopolitical_affiliation'].unique().tolist()

ownership_df

,ownership_share,system_ID,year_RFS,year_WFS,tot_owners,geopol_cow,gov_priv,geopolitical_affiliation,geopolitical_players
1116,1.000000,2000.1,2000.0,2025,1.0,USA,PRIV,United States,1.0
1117,1.000000,2000.10,2000.0,2025,1.0,DEN,PRIV,Denmark,1.0
1118,1.000000,2000.11,2000.0,2025,1.0,OTH,PRIV,Finland,0.0
1120,1.000000,2000.12,2000.0,2025,1.0,SWD,OTH,Sweden,1.0
1121,1.000000,2000.13,2000.0,2025,1.0,USA,PRIV,United States,1.0
...,...,...,...,...,...,...,...,...,...
5903,0.500000,2023.9,2023.0,2048,2.0,USA,PRIV,United States,1.0
5913,1.000000,2024.4,2024.0,2049,1.0,OTH,PRIV,Jamaica,0.0
5917,0.333333,2024.9,2024.0,2049,3.0,USA,PRIV,United States,1.0
5918,0.333333,2024.9,2024.0,2049,3.0,OTH,GOV,Oman,0.0


In [5]:
ownership_df['system_ID'] = ownership_df['system_ID'].astype(str)
nodes['system_ID'] = nodes['system_ID'].astype(str)
links['system_ID'] = links['system_ID'].astype(str)
# Group ownership data by system_ID to collect all affiliations per system
owners_grouped = ownership_df.groupby('system_ID').agg({
    'tot_owners': 'first',  # assumes tot_owners is consistent per system_ID
    'geopolitical_affiliation': list
}).reset_index()

# Merge with nodes
nodes = nodes.merge(owners_grouped, on='system_ID', how='left')
nodes['ownership_share'] = 1 / nodes['tot_owners']
nodes

,Node ID,Name,Longitude,Latitude,Hop_ID,system_ID,Country,tot_owners,geopolitical_affiliation,ownership_share
0,pt_54.5747_11.9311,"Gedser, Denmark",11.9311,54.5747,1,2000.1,Denmark,1.0,[United States],1.000000
1,pt_54.0833_12.1333,"Rostock, Germany",12.1333,54.0833,2,2000.1,Germany,1.0,[United States],1.000000
2,pt_59.4372_24.745,"Tallinn, Estonia",24.7450,59.4372,3,2000.11,Estonia,1.0,[Finland],1.000000
3,pt_60.1708_24.9375,"Helsinki, Finland",24.9375,60.1708,4,2000.11,Finland,1.0,[Finland],1.000000
4,pt_54.5_11.2167,"Puttgarden, Germany",11.2167,54.5000,5,2000.12,Germany,1.0,[Sweden],1.000000
...,...,...,...,...,...,...,...,...,...,...
1591,pt_38.1157_13.3613,"Palermo, Italy",13.3613,38.1157,1592,2024.9,Italy,3.0,"[United States, Oman, Italy]",0.333333
1592,pt_41.8931_12.4828,"Rome, Italy",12.4828,41.8931,1593,2024.9,Italy,3.0,"[United States, Oman, Italy]",0.333333
1593,pt_41.0038_9.6149,"Golfo Aranci, Italy",9.6149,41.0038,1594,2024.9,Italy,3.0,"[United States, Oman, Italy]",0.333333
1594,pt_42.7_9.4494,"Bastia, France",9.4494,42.7000,1595,2024.9,France,3.0,"[United States, Oman, Italy]",0.333333


In [6]:
links = links.merge(owners_grouped, on='system_ID', how='left')
links['ownership_share'] = 1 / links['tot_owners']
links

,Edge ID,Start Node,Start Name,Start Coordinates,End Node,End Name,End Coordinates,distance_km,system_ID,start_point_idx,end_point_idx,tot_owners,geopolitical_affiliation,ownership_share
0,pt_54.5747_11.9311-pt_54.0833_12.1333,pt_54.5747_11.9311,"Gedser, Denmark","(11.9311, 54.5747)",pt_54.0833_12.1333,"Rostock, Germany","(12.1333, 54.0833)",56.257798,2000.1,1,2,1.0,[United States],1.000000
1,pt_59.4372_24.745-pt_60.1708_24.9375,pt_59.4372_24.745,"Talinn, Estonia","(24.745, 59.4372)",pt_60.1708_24.9375,"Helsinki, Finland","(24.9375, 60.1708)",82.440640,2000.11,3,4,1.0,[Finland],1.000000
2,pt_54.5_11.2167-pt_54.6631_11.36,pt_54.5_11.2167,"Puttgarden, Germany","(11.2167, 54.5)",pt_54.6631_11.36,"Rõdbyhavn, Denmark","(11.36, 54.6631)",20.383229,2000.12,5,6,1.0,[Sweden],1.000000
3,pt_39.603509_-74.338118-pt_32.304027_-64.756309,pt_39.603509_-74.338118,"Tuckerton, United States","(-74.3381182, 39.6035094)",pt_32.304027_-64.756309,"Bermuda, Bermuda","(-64.7563086, 32.3040273)",1183.232537,2000.13,7,8,1.0,[United States],1.000000
4,pt_32.304027_-64.756309-pt_-3.730451_-38.525,pt_32.304027_-64.756309,"Bermuda, Bermuda","(-64.7563086, 32.3040273)",pt_-3.730451_-38.525,"Fortaleza, Brazil","(-38.525, -3.7304512)",4856.707511,2000.13,8,9,1.0,[United States],1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1305,pt_38.1157_13.3613-grid_38.9_14.7,pt_38.1157_13.3613,"Palermo, Italy","(13.3613, 38.1157)",grid_38.9_14.7,grid_38.9_14.7,"(14.7, 38.9)",145.647549,2024.9,1592,1580,3.0,"[United States, Oman, Italy]",0.333333
1306,pt_41.8931_12.4828-grid_41.3_11.7,pt_41.8931_12.4828,"Rome, Italy","(12.4828, 41.8931)",grid_41.3_11.7,grid_41.3_11.7,"(11.7, 41.3)",92.727698,2024.9,1593,1581,3.0,"[United States, Oman, Italy]",0.333333
1307,pt_41.0038_9.6149-grid_41.9_10.8,pt_41.0038_9.6149,"Golfo Aranci, Italy","(9.6149, 41.0038)",grid_41.9_10.8,grid_41.9_10.8,"(10.8, 41.9)",140.399840,2024.9,1594,1582,3.0,"[United States, Oman, Italy]",0.333333
1308,pt_42.7_9.4494-grid_43.8_8.4,pt_42.7_9.4494,"Bastia, France","(9.4494, 42.7)",grid_43.8_8.4,grid_43.8_8.4,"(8.4, 43.8)",148.984594,2024.9,1595,1584,3.0,"[United States, Oman, Italy]",0.333333


Next, the relevant affliations are processed, and the folders created and the iterative traffic assignments are run, with each ownership removed from the network and results saved into the output folder.

In [7]:
# Ensure output folder exists
output_folder = 'ownership_removal_scenarios'
os.makedirs(output_folder, exist_ok=True)

def process_and_save(df, df_name, affiliation):
    if isinstance(affiliation, float) and math.isnan(affiliation):
        return
    
    safe_affiliation = str(affiliation).replace(' ', '_').replace('/', '_').replace(',', '')
    mask = (
        (df['tot_owners'] <= 5) &
        (df['geopolitical_affiliation'].apply(
            lambda x: affiliation in x if isinstance(x, list) else False
        ))
    )
    
    # Rows that would remain (not removed)
    remaining = df[~mask].copy()
    filename = f"{output_folder}/{df_name}_removed_{safe_affiliation}.csv"
    remaining.to_csv(filename, index=False)

# Process each affiliation for nodes and links
for affiliation in unique_affiliations:
    process_and_save(nodes, 'nodes', affiliation)
    process_and_save(links, 'links', affiliation)

In [8]:
# 1. Define paths and ensure directories exist
scenario_folder = r"C:\Users\Dean\Downloads\IfW\ownership_removal_scenarios"
output_folder = r"C:\Users\Dean\Downloads\IfW\ownership_analysis_results"
os.makedirs(scenario_folder, exist_ok=True)  # Fix for FileNotFoundError
os.makedirs(output_folder, exist_ok=True)

# 2. Identify existing scenarios and results
all_files = os.listdir(scenario_folder)
existing_outputs = os.listdir(output_folder)

# 3. Find processed affiliations (all 4 files must exist)
processed = set()
for f in existing_outputs:
    if f.startswith("final_df_removed_"):
        affiliation = f.split("final_df_removed_")[1].replace(".csv", "")
        required_files = [
            f"final_df_removed_{affiliation}.csv",
            f"merged_data_removed_{affiliation}.csv",
            f"nodes_df_removed_{affiliation}.csv",
            f"links_df_removed_{affiliation}.csv"
        ]
        if all(x in existing_outputs for x in required_files):
            processed.add(affiliation)

# 4. Find all valid affiliations from input files
affiliations = set()
for f in all_files:
    if f.startswith("nodes_removed_"):
        affiliation = f.split("nodes_removed_")[1].replace(".csv", "")
        # Verify both nodes and links files exist
        if f"links_removed_{affiliation}.csv" in all_files:
            affiliations.add(affiliation)

# 5. Create processing queue
queue = [aff for aff in affiliations if aff not in processed]
total = len(affiliations)
completed = len(processed)
remaining = len(queue)

print(f"📊 Scenario Statistics:")
print(f"- Total scenarios identified: {total}")
print(f"- Already completed: {completed}")
print(f"- Remaining to process: {remaining}\n")

# 6. Process with progress bar
for affiliation in queue:
    try:
        print(f"Processing Affiliation: {affiliation}")
        
        # Load scenario data
        nodes = pd.read_csv(os.path.join(scenario_folder, f"nodes_removed_{affiliation}.csv"))
        links = pd.read_csv(os.path.join(scenario_folder, f"links_removed_{affiliation}.csv"))
        
        # Process scenario (assuming process_scenario is defined elsewhere)
        final_df, merged_data, nodes_df, links_df = process_scenario(nodes, links, od_data)
        
        # Save outputs
        final_df.to_csv(os.path.join(output_folder, f"final_df_removed_{affiliation}.csv"), index=False)
        merged_data.to_csv(os.path.join(output_folder, f"merged_data_removed_{affiliation}.csv"), index=False)
        nodes_df.to_csv(os.path.join(output_folder, f"nodes_df_removed_{affiliation}.csv"), index=False)
        links_df.to_csv(os.path.join(output_folder, f"links_df_removed_{affiliation}.csv"), index=False)
        
        print(f"✅ Finished processing: {affiliation}")
            
    except Exception as e:
        print(f"\n⚠️ Error processing {affiliation}: {str(e)}")

print("✅ Processing complete! All scenarios processed.")

📊 Scenario Statistics:
- Total scenarios identified: 120
- Already completed: 120
- Remaining to process: 0

✅ Processing complete! All scenarios processed.


In [11]:
# Explode the geopolitical_affiliation lists to create separate rows for each affiliation
exploded_affiliations = owners_grouped.explode('geopolitical_affiliation')

# Count unique system_IDs per affiliation
affiliation_counts = exploded_affiliations.groupby('geopolitical_affiliation')['system_ID'].nunique().reset_index()
affiliation_counts.columns = ['Affiliation', 'Unique System IDs Removed']

In [12]:
affiliation_counts

,Affiliation,Unique System IDs Removed
0,Algeria,2
1,Angola,3
2,Argentina,7
3,Armenia,2
4,Australia,11
...,...,...
115,United States,70
116,Uruguay,5
117,Venezuela,4
118,Vietnam,5


In [14]:
# Define output file path
output_excel_path = r'C:\Users\Dean\Downloads\IfW\ownership_systems_removed.xlsx'

# Save DataFrame to Excel
affiliation_counts.to_excel(output_excel_path, index=False)

print(f"Combined results saved to: {output_excel_path}")

Combined results saved to: C:\Users\Dean\Downloads\IfW\ownership_systems_removed.xlsx
